In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from tqdm.auto import tqdm
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Category mapping
category_map = {
    'Summaries': ['summaries'],
    'English': ['english_1', 'english_2'],
    'Math': ['math_1', 'math_2'],
    'Coding': ['coding_1', 'coding_2', 'coding_3'],
    'Reasoning': ['reasoning_1', 'reasoning_2', 'reasoning_3']
}

# Load Data
with open("df_processed.pkl", "rb") as f:
    df_processed = pickle.load(f)

# Load original CSV to extract Source info
df = pd.read_csv("Training/ranked_responses_final.csv")

# Fix: drop duplicates so each prompt maps to a unique Source
source_map = df.drop_duplicates(subset="Prompt")[["Prompt", "Source"]].set_index("Prompt")["Source"]

# Map Source into processed DataFrame
df_processed["Source"] = df_processed["prompt"].map(source_map)

# Drop rows that failed to map (just in case)
df_processed = df_processed.dropna(subset=["Source"])

In [ ]:
# Training by category
for category, sources in category_map.items():
    print(f"\n===== Training for category: {category} =====")
    category_df = df_processed[df_processed["Source"].isin(sources)]
    X = np.vstack(category_df["embedding"])
    y = pd.DataFrame(category_df["scores"].tolist(), columns=[
        "openai/gpt-4o", "anthropic/claude-3.5-sonnet",
        "deepseek/deepseek-chat", "perplexity/sonar"])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = MultiOutputRegressor(
        RandomForestRegressor(n_jobs=4, random_state=42),
        n_jobs=4
    )

    param_grid = {
        "estimator__n_estimators": [100],
        "estimator__max_depth": [5, 10],
        "estimator__min_samples_split": [2, 5],
    }

    grid_search = GridSearchCV(
        model,
        param_grid,
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=1,
        verbose=3,
        return_train_score=True
    )

    total_combinations = np.prod([len(v) for v in param_grid.values()])
    total_iterations = total_combinations * 5

    class TqdmJoblib:
        def __init__(self, tqdm_object): self.tqdm_object = tqdm_object
        def __enter__(self):
            self.original_callback = joblib.parallel.BatchCompletionCallBack
            tqdm_obj = self.tqdm_object
            class TqdmBatchCompletionCallBack(self.original_callback):
                def __call__(self, *args, **kwargs):
                    result = super().__call__(*args, **kwargs)
                    tqdm_obj.update(n=self.batch_size)
                    return result
            joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallBack
            return self
        def __exit__(self, exc_type, exc_val, exc_tb):
            joblib.parallel.BatchCompletionCallBack = self.original_callback

    with TqdmJoblib(tqdm(total=total_iterations, desc=f"{category} Grid Search")):
        grid_search.fit(X_train, y_train)

    best_clf = grid_search.best_estimator_
    joblib.dump(best_clf, f"best_RF_{category}.pkl")
    print(f"Saved model for {category} as best_RF_{category}.pkl")

    y_pred = best_clf.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"\n{category} Regression Metrics:")
    print(f"MSE: {mse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

    # Plot training curve (mean RMSE over iterations if available)
    if hasattr(grid_search.best_estimator_.estimator_, "oob_improvement_"):
        plt.plot(grid_search.best_estimator_.estimator_.oob_improvement_)
        plt.title(f"{category} OOB Improvement Over Trees")
        plt.xlabel("Trees")
        plt.ylabel("OOB Improvement")
        plt.show()

    def convert_to_ranking(scores):
        sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return {model: rank + 1 for rank, (model, _) in enumerate(sorted_models)}

    model_names = list(y.columns)
    actual_ranks, predicted_ranks = [], []
    actual_best_models, predicted_best_models = [], []

    for i in range(len(X_test)):
        actual_scores = dict(zip(model_names, y_test.iloc[i].values))
        predicted_scores = dict(zip(model_names, y_pred[i]))
        actual_ranking = convert_to_ranking(actual_scores)
        predicted_ranking = convert_to_ranking(predicted_scores)
        for model in model_names:
            actual_ranks.append(actual_ranking[model])
            predicted_ranks.append(predicted_ranking[model])
        actual_best_models.append(min(actual_ranking, key=actual_ranking.get))
        predicted_best_models.append(min(predicted_ranking, key=predicted_ranking.get))

    # Classification on full rankings
    conf_mat = confusion_matrix(actual_ranks, predicted_ranks, labels=[1, 2, 3, 4])
    acc_score = accuracy_score(actual_ranks, predicted_ranks)
    precision_val = precision_score(actual_ranks, predicted_ranks, average='macro')
    recall_val = recall_score(actual_ranks, predicted_ranks, average='macro')
    f1_val = f1_score(actual_ranks, predicted_ranks, average='macro')

    print(f"\n{category} Classification Metrics:")
    print(f"Accuracy: {acc_score:.4f} | Precision: {precision_val:.4f} | Recall: {recall_val:.4f} | F1: {f1_val:.4f}")
    plt.figure(figsize=(6, 5))
    sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
                xticklabels=[1, 2, 3, 4], yticklabels=[1, 2, 3, 4])
    plt.title(f"{category} Confusion Matrix (Full Rankings)")
    plt.xlabel("Predicted Rank")
    plt.ylabel("Actual Rank")
    plt.tight_layout()
    plt.show()

    # Evaluation for rank 1 only
    acc_score = accuracy_score(actual_best_models, predicted_best_models)
    precision_val = precision_score(actual_best_models, predicted_best_models, average='macro')
    recall_val = recall_score(actual_best_models, predicted_best_models, average='macro')
    f1_val = f1_score(actual_best_models, predicted_best_models, average='macro')
    conf_mat_best = confusion_matrix(actual_best_models, predicted_best_models, labels=model_names)

    print(f"\n{category} Evaluation for Predicting Best Model (Rank 1 Only):")
    print(f"Accuracy: {acc_score:.4f} | Precision: {precision_val:.4f} | Recall: {recall_val:.4f} | F1: {f1_val:.4f}")
    plt.figure(figsize=(7, 6))
    sns.heatmap(conf_mat_best, annot=True, fmt='d', cmap='Blues',
                xticklabels=model_names, yticklabels=model_names)
    plt.title(f"{category} Confusion Matrix (Rank 1 Only)")
    plt.xlabel("Predicted Best Model")
    plt.ylabel("Actual Best Model")
    plt.tight_layout()
    plt.show()